# 中山醫學大學附設醫院 SmartCoder 公開展示

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dechnology/csh-smartcoder-demo/blob/main/csh_smartcoder_api_demo.ipynb)

選「執行階段」→「全部執行」即可完成展示，不需輸入 API key 或設定環境變數。

> 請只使用下方內建的合成案例。請勿輸入真實病歷、姓名、身分證號、病歷號或其他個人資料。

公開展示用 `X-API-Key` 已內建在 Notebook，直接呼叫中山醫 SmartCoder 正式服務。每次請求都會從原始病歷依序執行病歷整理、3 輪實體辨識、術語連結與 SNOMED 驗證；展示憑證可由維運端隨時更換。

In [1]:
from uuid import uuid4
import json
import time

import requests

BASE_URL = "https://fhircsh.itri-nlp.tw/code_api/smartcoder"
PUBLIC_DEMO_X_API_KEY = "csh-public-demo-v1"  # 可公開、可更換的展示 token
COLAB_ORIGIN = "https://colab.research.google.com"
CODE_URL = f"{BASE_URL}/api/v1/snomed/coding"
RESULT_URL = f"{BASE_URL}/api/v1/snomed/results/{{request_id}}"
HEADERS = {
    "Content-Type": "application/json",
    "X-API-Key": PUBLIC_DEMO_X_API_KEY,
    "X-Hospital-Slug": "csh",
    "Origin": COLAB_ORIGIN,
}

print("中山醫 SmartCoder 正式服務已就緒：", BASE_URL)

中山醫 SmartCoder 正式服務已就緒： https://fhircsh.itri-nlp.tw/code_api/smartcoder


In [2]:
def assert_colab_cors(response):
    actual = response.headers.get("Access-Control-Allow-Origin")
    assert actual == COLAB_ORIGIN, f"CORS 不符：{actual!r}"


def safe_result(payload):
    metadata = payload.get("processing_metadata") or {}
    public_metadata = {
        key: metadata.get(key)
        for key in ("snomed_version", "pipeline_version", "vote_attempts", "confidence_method")
    }
    return {
        "request_id": payload.get("request_id"),
        "polished_clinical_note": payload.get("polished_clinical_note"),
        "snomed_codings": payload.get("snomed_codings", []),
        "processing_metadata": public_metadata,
    }


request_id = str(uuid4())
demo_case = {
    "request_id": request_id,
    "encounter_type": "outpatient",
    "raw_clinical_note": "患者胸痛持續兩週，否認咳嗽。",
    "output_format": "simple",
}

started_at = time.perf_counter()
post_response = requests.post(
    CODE_URL, headers=HEADERS, json=demo_case, timeout=240
)
elapsed_seconds = time.perf_counter() - started_at
print(f"POST HTTP {post_response.status_code}｜完整流程 {elapsed_seconds:.2f} 秒")
post_response.raise_for_status()
assert_colab_cors(post_response)
post_payload = post_response.json()
assert post_payload.get("request_id") == request_id, "POST request_id 不一致"
assert (post_payload.get("polished_clinical_note") or "").strip(), "缺少整理後病歷"
assert post_payload.get("snomed_codings"), "POST 編碼結果不得為空"
metadata = post_payload.get("processing_metadata") or {}
assert str(metadata.get("pipeline_version", "")).endswith("|txt_ner"), "不是完整 TXT_NER 流程"
assert metadata.get("vote_attempts") == 3, "NER 必須執行 3 輪"
assert all(item.get("source") == ["TXT_NER"] for item in post_payload["snomed_codings"]), "結果含有非 TXT_NER 來源"
assert any(item.get("concept_id") == "29857009" for item in post_payload["snomed_codings"]), "未辨識出胸痛 SNOMED 概念"

get_response = requests.get(
    RESULT_URL.format(request_id=request_id), headers=HEADERS, timeout=60
)
print("GET HTTP", get_response.status_code)
get_response.raise_for_status()
assert_colab_cors(get_response)
get_payload = get_response.json()
assert get_payload.get("request_id") == request_id, "GET request_id 不一致"
assert get_payload.get("status") == "completed", "GET 任務狀態不是 completed"
assert (get_payload.get("response") or {}).get("snomed_codings") == post_payload.get("snomed_codings"), "GET 與 POST 編碼結果不一致"

print(json.dumps(safe_result(post_payload), ensure_ascii=False, indent=2))
print("\n驗收通過：原始病歷已完成整理、3 輪 NER、SNOMED 連結、同 request_id GET 與 Colab CORS。")

POST HTTP 200｜完整流程 17.26 秒
GET HTTP 200
{
  "request_id": "c47e52f1-0f38-4362-b199-617e019b13b3",
  "polished_clinical_note": "## Present Illness\n\nThe patient has been experiencing chest pain for the past two weeks. The patient denies having a cough.",
  "snomed_codings": [
    {
      "concept_id": "29857009",
      "term": "Chest pain",
      "category": "finding",
      "tui": "T033",
      "confidence": 1.0,
      "source": [
        "TXT_NER"
      ]
    }
  ],
  "processing_metadata": {
    "snomed_version": "2026-07-01",
    "pipeline_version": "api-version-0.1.0|txt_ner",
    "vote_attempts": 3,
    "confidence_method": "txt_ner_assertion_filtered_vote_support_min_2"
  }
}

驗收通過：原始病歷已完成整理、3 輪 NER、SNOMED 連結、同 request_id GET 與 Colab CORS。
